# z301 – Preprocesamiento
**Grupo 3: Banegas - Marín - Mengoni - Rey**

Lee `sell-in.txt.gz` → agrupa → aplica estrategia de nulos → densifica → escribe parquet a VM + GCS.

| Palanca | Opciones | Default |
|---------|----------|---------|
| `group_mode` | A (cliente-producto) / B (producto) | B |
| `missing_strategy` | zero / null | zero |
| `densify_strategy` | full / lifecycle | full |

In [ ]:
import yaml, time
from pathlib import Path
import polars as pl
from google.cloud import storage as gcs

with open('../pipe_py/config.yaml') as f:
    CFG = yaml.safe_load(f)

# --- palancas (modificar acá para experimentar) ---
CFG['preproc']['group_mode']       = 'B'
CFG['preproc']['missing_strategy'] = 'zero'
CFG['preproc']['densify_strategy'] = 'full'
# ---------------------------------------------------

pr = CFG['preproc']
grupo = f"{pr['group_mode']}_{pr['missing_strategy']}_{pr['densify_strategy']}"
print(f'Configuración: {grupo}')

In [ ]:
# Leer raw
t0 = time.time()
lf = pl.scan_csv(CFG['paths']['raw_sellin'], separator='\t', infer_schema_length=50_000)
print('Schema:', lf.schema)

In [ ]:
# Agrupación
group_mode = pr['group_mode']
keys = ['customer_id','product_id','periodo'] if group_mode=='A' else ['product_id','periodo']
lf = lf.group_by(keys).agg(pl.col('tn').sum())
print('Keys:', keys)

In [ ]:
# Nulos
if pr['missing_strategy'] == 'zero':
    lf = lf.with_columns(pl.col('tn').fill_null(0.0))
print('Missing strategy:', pr['missing_strategy'])

In [ ]:
# Densificación full
if pr['densify_strategy'] == 'full':
    periodos = [p for p in range(pr['periodo_min'], pr['periodo_max']+1) if 1 <= (p%100) <= 12]
    id_cols = ['customer_id','product_id'] if group_mode=='A' else ['product_id']
    entidades = lf.select(id_cols).unique()
    grilla = entidades.join(pl.LazyFrame({'periodo': periodos}), how='cross')
    lf = grilla.join(lf, on=id_cols+['periodo'], how='left').with_columns(pl.col('tn').fill_null(0.0))

print('Densify:', pr['densify_strategy'], '| periodos:', pr['periodo_min'], '→', pr['periodo_max'])

In [ ]:
# Agrupacion_ID
if group_mode == 'A':
    lf = lf.with_columns((pl.col('customer_id')*100_000 + pl.col('product_id')).alias('Agrupacion_ID'))
else:
    lf = lf.with_columns(pl.col('product_id').alias('Agrupacion_ID'))

In [ ]:
# Escribir parquet
ruta_out = Path(CFG['paths']['preproc_out']) / f'preprocesado_{grupo}.parquet'
ruta_out.parent.mkdir(parents=True, exist_ok=True)
lf.sink_parquet(str(ruta_out), compression='snappy')

df_check = pl.read_parquet(str(ruta_out))
print(f'Filas: {len(df_check):,} | Columnas: {df_check.columns}')
df_check.head()

In [ ]:
# EDA rápido
print('Distribución tn:')
print(df_check['tn'].describe())
print(f'\nProductos únicos: {df_check["product_id"].n_unique()}')
print(f'Períodos: {sorted(df_check["periodo"].unique().to_list())}')

In [ ]:
# Subir a GCS
g = CFG['gcs']
client = gcs.Client()
bucket = client.bucket(g['bucket'])
blob_path = f"{g['prefix_preproc']}/preprocesado_{grupo}.parquet"
blob = bucket.blob(blob_path)
blob.upload_from_filename(str(ruta_out))
print(f"Subido → gs://{g['bucket']}/{blob_path}")
print(f'Tiempo total: {time.time()-t0:.1f}s')